# Baseline Analysis

In [1]:
import polars as pl

from social_groups.reporting.analysis_columns import AnalysisColumn
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)


In [2]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

In [3]:
from social_groups.analysis.definitions import defs

baseline_frame = defs.load_fn().load_asset_value("baseline")

/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/.venv/lib/python3.11/site-packages/dagster/_config/pythonic_config/typing_utils.py:111: UserWarning: Field name "extension" in "PolarsParquetIOManager" shadows an attribute in parent "BasePolarsUPathIOManager"
  return super().__new__(cls, name, bases, namespaces, **kwargs)
2026-02-21 19:35:56 +0800 - dagster - DEBUG - system - Loading file from: /Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/analysis/dagster/baseline.parquet using PolarsParquetIOManager...


In [4]:
baseline_frame.head()

id,run_id,question_id,phoenix_span_url,run_identifier,final_answer,original_question_id,category,question,answer_string,model_name
i64,i64,i64,str,str,str,i64,str,str,str,str
0,1,0,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's see. The q…",6622,"""health""","""Q: What stable isotope is comm…","""J""","""Qwen/Qwen3-0.6B"""
1,1,1,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's see. The q…",6116,"""health""","""Q: A 2-month-old female is bro…","""F""","""Qwen/Qwen3-0.6B"""
2,1,2,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",1556,"""law""","""Q: One evening, an undercover …","""B""","""Qwen/Qwen3-0.6B"""
3,1,3,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's see. The q…",10297,"""physics""","""Q: Kirkwood gaps are observed …","""F""","""Qwen/Qwen3-0.6B"""
4,1,4,"""http://localhost:6006/projects…","""2026-01-28-11-18-38 - heteroge…","""<think> Okay, let's try to fig…",1945,"""law""","""Q: A wealthy woman often wore …","""E""","""Qwen/Qwen3-0.6B"""


### Number of unparsable answers

In [5]:
(
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .group_by("model_name")
    .agg(
        no_null=pl.col(AnalysisColumn.parsed_answer.value)
        .str.starts_with("___")
        .not_()
        .sum(),
        null_percentage=(
                pl.col(AnalysisColumn.parsed_answer.value).str.starts_with("___").mean()
                * 100
        ).round(2),
    )
)

model_name,no_null,null_percentage
str,u32,f64
"""Qwen/Qwen3-4B""",88,12.0
"""Qwen/Qwen3-14B""",93,7.0
"""Qwen/Qwen3-0.6B""",90,10.0


### Accuracy per Model

In [6]:
(
    baseline_frame.with_columns(
        parser(pl.col("final_answer")).alias(AnalysisColumn.parsed_answer.value)
    )
    .with_columns(
        is_correct=comparer(
            pl.col(AnalysisColumn.parsed_answer.value), pl.col("answer_string")
        )
    )
    .group_by("model_name")
    .agg(accuracy=pl.col("is_correct").mean())
    .sort(pl.col("model_name").str.extract(r"-(\d+\.?\d*)B", 1).cast(pl.Float64))
)

model_name,accuracy
str,f64
"""Qwen/Qwen3-0.6B""",0.35
"""Qwen/Qwen3-4B""",0.59
"""Qwen/Qwen3-14B""",0.64
